# Notebook 7 – Observatory Data Pipeline

## Objective

Merge the cleaned metabolic health, food system, and demographic datasets into a single master observatory dataset for the Pacific Nutrition Transition Observatory.

**Inputs:**

- Adult obesity
- Adult diabetes
- Adult hypertension
- Adult BMI distribution
- FAOSTAT food system indicators
- UN population indicators

**Output:**

- `processed_data/observatory_master.csv`

This dataset will support downstream statistical analysis, visualization, and dashboard development.

## 1. Initialize project folders

In [ ]:
import pandas as pd
import os

os.makedirs("processed_data", exist_ok=True)

print("Project folders ready.")
print("Current files:")
print(os.listdir())

## 2. Upload and organize processed input files

In [ ]:
# ============================================================
# Section 2. Upload and Organize Processed Input Files
# ============================================================

import os
import shutil

expected_files = [
    "adult_obesity.csv",
    "adult_diabetes.csv",
    "adult_hypertension.csv",
    "adult_bmi_distribution.csv",
    "food_system_indicators.csv",
    "population.csv"
]

os.makedirs("processed_data", exist_ok=True)

for file in expected_files:
    if os.path.exists(f"processed_data/{file}"):
        print(f"Already in processed_data: {file}")
    elif os.path.exists(file):
        shutil.move(file, f"processed_data/{file}")
        print(f"Moved into processed_data: {file}")
    else:
        print(f"Missing: {file}")

print("\nFiles in processed_data:")
print(sorted(os.listdir("processed_data")))

Already in processed_data: adult_obesity.csv
Already in processed_data: adult_diabetes.csv
Already in processed_data: adult_hypertension.csv
Already in processed_data: adult_bmi_distribution.csv
Already in processed_data: food_system_indicators.csv
Already in processed_data: population.csv

Files in processed_data:
['adult_bmi_distribution.csv', 'adult_diabetes.csv', 'adult_hypertension.csv', 'adult_obesity.csv', 'food_system_indicators.csv', 'population.csv']


## 3. Load processed datasets

Load each standardized dataset into a pandas DataFrame. These datasets were created by the preceding ETL notebooks and serve as the inputs to the Pacific Nutrition Transition Observatory.

In [ ]:
import pandas as pd

obesity = pd.read_csv("processed_data/adult_obesity.csv")
diabetes = pd.read_csv("processed_data/adult_diabetes.csv")
hypertension = pd.read_csv("processed_data/adult_hypertension.csv")
bmi = pd.read_csv("processed_data/adult_bmi_distribution.csv")
population = pd.read_csv("processed_data/population.csv")
food = pd.read_csv("processed_data/food_system_indicators.csv")
print("Datasets loaded successfully.\n")

print("Obesity:", obesity.shape)
print("Diabetes:", diabetes.shape)
print("Hypertension:", hypertension.shape)
print("BMI:", bmi.shape)
print("Food System:", food.shape)
print("Population:", population.shape)

Datasets loaded successfully.

Obesity: (720, 4)
Diabetes: (528, 5)
Hypertension: (480, 8)
BMI: (720, 10)
Food System: (271, 7)
Population: (1184, 8)


## 4. Verify merge keys

Before merging the datasets, verify that each dataset contains the expected country and year fields. Consistent geographic and temporal keys help ensure that records align correctly across health, food system, and population sources.

In [ ]:
datasets = {
    "Obesity": obesity,
    "Diabetes": diabetes,
    "Hypertension": hypertension,
    "BMI": bmi,
    "Food System": food,
    "Population": population
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * len(name))
    print(df.columns.tolist())


Obesity
-------
['ISO3', 'Country', 'Year', 'Obesity_Pct']

Diabetes
--------
['ISO3', 'Country', 'Year', 'Diabetes_Pct', 'Diabetes_Treated_Pct']

Hypertension
------------
['ISO3', 'Country', 'Year', 'Hypertension_Pct', 'Hypertension_Diagnosed_Pct', 'Hypertension_Treated_Pct', 'Hypertension_Controlled_Pct', 'Untreated_Stage2_Pct']

BMI
---
['ISO3', 'Country', 'Year', 'Underweight_Pct', 'HealthyWeight_Pct', 'Overweight_Pct', 'Obesity_Pct', 'Class1_Obesity_Pct', 'Class2_Obesity_Pct', 'Morbid_Obesity_Pct']

Food System
-----------
['ISO3', 'Country', 'Year', 'Dietary_Energy_kcal', 'Fat_g', 'Protein_g', 'Food_Imports_Export_Value_Pct']

Population
----------
['ISO3', 'Country', 'Year', 'Population', 'Population_Thousands', 'Population_Density', 'Median_Age', 'Population_Growth_Rate']


## 5. Check duplicate obesity fields

The obesity-only dataset and the BMI distribution dataset both contain `Obesity_Pct`. Before merging, confirm whether these fields are identical.

In [ ]:
obesity_check = bmi[["ISO3", "Country", "Year", "Obesity_Pct"]].merge(
    obesity[["ISO3", "Country", "Year", "Obesity_Pct"]],
    on=["ISO3", "Country", "Year"],
    how="outer",
    suffixes=("_BMI", "_Obesity")
)

obesity_check["Difference"] = (
    obesity_check["Obesity_Pct_BMI"] - obesity_check["Obesity_Pct_Obesity"]
).abs()

print("Rows:", len(obesity_check))
print("Missing values:")
print(obesity_check.isna().sum())

print("\nMaximum difference:")
print(obesity_check["Difference"].max())

obesity_check.head()

Rows: 765
Missing values:
ISO3                    0
Country                 0
Year                    0
Obesity_Pct_BMI        45
Obesity_Pct_Obesity    45
Difference             90
dtype: int64

Maximum difference:
75.09613089620237


,ISO3,Country,Year,Obesity_Pct_BMI,Obesity_Pct_Obesity,Difference
0,ASM,American Samoa,1980,0.541396,52.256046,51.714650
1,ASM,American Samoa,1981,0.553142,53.427460,52.874318
2,ASM,American Samoa,1982,0.564962,54.613544,54.048582
3,ASM,American Samoa,1983,0.576719,55.793793,55.217074
4,ASM,American Samoa,1984,0.588321,56.962180,56.373859


## 6. Standardize BMI distribution units

The BMI distribution file stores prevalence values as proportions, while the other health datasets use percentage units. Convert BMI distribution prevalence fields to percentages before merging.

In [ ]:
bmi_percent_columns = [
    "Underweight_Pct",
    "HealthyWeight_Pct",
    "Overweight_Pct",
    "Obesity_Pct",
    "Class1_Obesity_Pct",
    "Class2_Obesity_Pct",
    "Morbid_Obesity_Pct"
]

for col in bmi_percent_columns:
    bmi[col] = bmi[col] * 100

obesity_check = bmi[["ISO3", "Country", "Year", "Obesity_Pct"]].merge(
    obesity[["ISO3", "Country", "Year", "Obesity_Pct"]],
    on=["ISO3", "Country", "Year"],
    how="inner",
    suffixes=("_BMI", "_Obesity")
)

obesity_check["Difference"] = (
    obesity_check["Obesity_Pct_BMI"] - obesity_check["Obesity_Pct_Obesity"]
).abs()

print("Rows compared:", len(obesity_check))
print("Maximum difference:", obesity_check["Difference"].max())

obesity_check.head()

Rows compared: 675
Maximum difference: 1.8867641674608961


,ISO3,Country,Year,Obesity_Pct_BMI,Obesity_Pct_Obesity,Difference
0,ASM,American Samoa,1980,54.139611,52.256046,1.883565
1,ASM,American Samoa,1981,55.314224,53.427460,1.886764
2,ASM,American Samoa,1982,56.496159,54.613544,1.882615
3,ASM,American Samoa,1983,57.671857,55.793793,1.878064
4,ASM,American Samoa,1984,58.832065,56.962180,1.869885


## 7. Merge processed datasets

Merge the standardized datasets into a single observatory master table.

The BMI distribution dataset is used as the base table because it contains the broadest weight-category structure. The merge uses `ISO3` and `Year` as stable shared keys because country names differ slightly across source systems.

In [20]:
# ============================================================
# Section 7. Merge processed datasets
# ============================================================

# The BMI distribution file provides the canonical obesity percentage
# for the master dataset.

population_merge = population.drop(columns=["Country"])
food_merge = food.drop(columns=["Country"])

observatory_master = bmi.merge(
    diabetes.drop(columns=["Country"]),
    on=["ISO3", "Year"],
    how="left"
).merge(
    hypertension.drop(columns=["Country"]),
    on=["ISO3", "Year"],
    how="left"
).merge(
    food_merge,
    on=["ISO3", "Year"],
    how="left"
).merge(
    population_merge,
    on=["ISO3", "Year"],
    how="left"
)

print("Master dataset shape:", observatory_master.shape)
print("\nColumns:")
print(observatory_master.columns.tolist())

observatory_master.head()

Master dataset shape: (720, 26)

Columns:
['ISO3', 'Country', 'Year', 'Underweight_Pct', 'HealthyWeight_Pct', 'Overweight_Pct', 'Obesity_Pct', 'Class1_Obesity_Pct', 'Class2_Obesity_Pct', 'Morbid_Obesity_Pct', 'Diabetes_Pct', 'Diabetes_Treated_Pct', 'Hypertension_Pct', 'Hypertension_Diagnosed_Pct', 'Hypertension_Treated_Pct', 'Hypertension_Controlled_Pct', 'Untreated_Stage2_Pct', 'Dietary_Energy_kcal', 'Fat_g', 'Protein_g', 'Food_Imports_Export_Value_Pct', 'Population', 'Population_Thousands', 'Population_Density', 'Median_Age', 'Population_Growth_Rate']


,ISO3,Country,Year,Underweight_Pct,HealthyWeight_Pct,Overweight_Pct,Obesity_Pct,Class1_Obesity_Pct,Class2_Obesity_Pct,Morbid_Obesity_Pct,...,Untreated_Stage2_Pct,Dietary_Energy_kcal,Fat_g,Protein_g,Food_Imports_Export_Value_Pct,Population,Population_Thousands,Population_Density,Median_Age,Population_Growth_Rate
0,ASM,American Samoa,1980,0.746896,15.316396,28.330742,54.139611,28.999122,16.019193,9.121296,...,NaN,NaN,NaN,NaN,NaN,32403.0,32.403,162.013,17.832,3.275
1,ASM,American Samoa,1981,0.715825,14.656437,27.924248,55.314224,29.209004,16.534990,9.570230,...,NaN,NaN,NaN,NaN,NaN,33584.0,33.584,167.923,18.047,3.880
2,ASM,American Samoa,1982,0.685943,14.005578,27.496787,56.496159,29.398000,17.060237,10.037922,...,NaN,NaN,NaN,NaN,NaN,34909.0,34.909,174.545,18.234,3.856
3,ASM,American Samoa,1983,0.657359,13.371328,27.054243,57.671857,29.562999,17.589228,10.519631,...,NaN,NaN,NaN,NaN,NaN,36271.0,36.271,181.355,18.422,3.800
4,ASM,American Samoa,1984,0.629968,12.760030,26.599569,58.832065,29.702321,18.117063,11.012682,...,NaN,NaN,NaN,NaN,NaN,37667.0,37.667,188.335,18.613,3.754


## 8. Master dataset quality assurance

Evaluate the integrated observatory dataset after merging metabolic health, food system, and demographic indicators. This section checks dataset structure, country and year coverage, missing values, and nutrition transition indicator availability.

In [21]:
# ============================================================
# Section 8. Master dataset quality assurance
# ============================================================

print(observatory_master.info())

print("\nRows:", len(observatory_master))
print("Columns:", len(observatory_master.columns))

print("\nCountries:")
print(sorted(observatory_master["Country"].unique()))

print("\nYears:")
print(observatory_master["Year"].min(), "to", observatory_master["Year"].max())

print("\nMissing values:")
print(observatory_master.isna().sum())

print("\nNutrition transition coverage:")
nutrition_cols = [
    "Dietary_Energy_kcal",
    "Fat_g",
    "Protein_g",
    "Food_Imports_Export_Value_Pct"
]

print(observatory_master[nutrition_cols].notna().sum())

print("\nMissing population rows:")
print(
    observatory_master[observatory_master["Population"].isna()]
    [["ISO3", "Country", "Year"]]
    .sort_values(["Country", "Year"])
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 26 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   ISO3                           720 non-null    object 
 1   Country                        720 non-null    object 
 2   Year                           720 non-null    int64  
 3   Underweight_Pct                720 non-null    float64
 4   HealthyWeight_Pct              720 non-null    float64
 5   Overweight_Pct                 720 non-null    float64
 6   Obesity_Pct                    720 non-null    float64
 7   Class1_Obesity_Pct             720 non-null    float64
 8   Class2_Obesity_Pct             720 non-null    float64
 9   Morbid_Obesity_Pct             720 non-null    float64
 10  Diabetes_Pct                   528 non-null    float64
 11  Diabetes_Treated_Pct           528 non-null    float64
 12  Hypertension_Pct               480 non-null    flo

## 9. Export observatory master dataset

Export the fully merged observatory dataset for dashboard development, statistical analysis, and presentation figures.

In [22]:
# ============================================================
# Section 9. Export observatory master dataset
# ============================================================

output_file = "processed_data/observatory_master.csv"

observatory_master.to_csv(output_file, index=False)

print("Export complete.")
print(f"File: {output_file}")
print(f"Rows: {len(observatory_master)}")
print(f"Columns: {len(observatory_master.columns)}")

Export complete.
File: processed_data/observatory_master.csv
Rows: 720
Columns: 26
